# 매니퓰레이터 활용 쓰레기 수거용 사륜구동 원격 제어 시스템

주최기관: 계명대학교 지방대학활성화사업단수행과제

기간: 2023.03.02 ~ 2023.06.08

## 1. 프로젝트 설명

### ⚙️ 역할
하드웨어 담당(재료 선정, 디자인, 조립, 무게중심 문제 해결), 제어 알고리즘 개발

### ⚙️ 개발환경
하드웨어 - 라즈베리파이 4B(4GB), MG996서보모터(360°), 15kg 디지털어시스 서보 모터(180°),25T 서보모터 알루미늄 혼(원형),

DC 12V 320RPM 마이크로 기어 모터, 전방향 자동차 휠 (80mm) X 4, 10kg 부하 자동차 섀시, 브라켓(모터 고정, 매니퓰레이터 프레임), 배터리

소프트웨어 - Raspberry Pi, Python, VNC Viewer, PuTTY, APP INVENTOR, CATIA

### 🔍 프로젝트 개요
계명대학교 지방대학활성화사업단에서 시행하는 캡스톤 디자인 경연대회(기업체 연계형)에 참여.

무선 조종이 가능한 사륜구동 플랫폼에 집게형 그리퍼가 장착된 로봇 팔을 장착, 쓰레기를 원격으로 수거가능한 로봇 제작.

사람의 접근이 위험하거나 불편한 장소의 쓰레기를 보다 안전하고 효율적으로 수거할 수 있도록 기여.

### 🔍 프로젝트 목표
스마트폰 앱 연동을 통해 로봇 차체의 구동과 로봇 팔 무선 제어 시스템 구현.

목표 지점 이동, 쓰레기 안정적 파지, 차체에 내장된 휴지통으로 수거, 로봇 이동 프로세스를 안정적으로 제어.

### 📊 주요성과 및 문제해결
차체 프레임 재료 아크릴에서 우드락, 하드보드지 2겹으로 변경하여 전체적인 무게를 감소시켜 -30°~ 60° 경사에서도 주행이 가능하게 구현.

주행용 DC모터 속도 및 차체 로봇 팔 결합부 위치를 조정해 가속 시 무게중심 잃는 현상을 해결해 주행 안정성 개선.

별도의 16채널 모터 드라이버 사용에서 라즈베리파이 내장 GPIO핀 직접연결로 변경해 불필요한 배선을 줄여 내부 공간 활용 효율 증가 및 구조 단순화.

### 🗂️ 부품 명세서(BOM)

<img src="images/BOM.png" width="1000">

## 2. 하드웨어 구성 요소

### 🤖 Robot 외관
<img src="images/robot.png" width="500">

<img src="images/manipulator.png" width="500">


<img src="images/end_effector.png" width="500">

<img src="images/frame.png" width="500">



<img src="images/modul.png" width="1000">

## 3. 데이터 통신 및 이동 제어 알고리즘

### ⚙️ 데이터 통신 프로토콜
<img src="images/data_network_al.png" width="1000">

### ⚙️ 로봇 이동 알고리즘
<img src="images/robot_move_al.png" width="1000">

## 4. 매니퓰레이터 동작 관련 코드(python 앱인벤터 연동 코드_원본)

In [ ]:
# HTTP 요청을 처리하기 위한 기본 웹 서버 모듈
from http.server import BaseHTTPRequestHandler, HTTPServer
# 라즈베리파이의 GPIO 핀을 제어하기 위한 모듈
import RPi.GPIO as GPIO
import time

# GPIO 설정
GPIO.setmode(GPIO.BCM)
GPIO.setwarnings(False)

# DC 모터 핀 목록
dc_pins = [0, 5, 6, 13, 19, 26, 16, 20]
for pin in dc_pins:
    GPIO.setup(pin, GPIO.OUT)

# 서보모터 핀 설정
SERVO_GRIP = 10  # 그립
SERVO_JOINT1 = 9
SERVO_JOINT2 = 11

GPIO.setup(SERVO_GRIP, GPIO.OUT)
GPIO.setup(SERVO_JOINT1, GPIO.OUT)
GPIO.setup(SERVO_JOINT2, GPIO.OUT)

# PWM 객체 초기화 (100Hz)
servo_grip = GPIO.PWM(SERVO_GRIP, 100)
servo_joint1 = GPIO.PWM(SERVO_JOINT1, 100)
servo_joint2 = GPIO.PWM(SERVO_JOINT2, 100)

# 서보 각도 → 듀티비 변환 함수
def angle_to_duty(angle):
    return ((angle * 0.01) + 0.5) * 10

# DC모터 정지
def stop_dc_motors():
    for pin in dc_pins:
        GPIO.output(pin, False)

# DC모터 상태 설정
def set_dc_motors(state_list):
    for pin, state in zip(dc_pins, state_list):
        GPIO.output(pin, state)

# 서보 모터 움직이기
def move_servos(angle1, angle2, delay=0.2):
    # 각도를 PWM 듀티사이클로 변환
    duty1 = angle_to_duty(angle1)  # 관절1 각도
    duty2 = angle_to_duty(angle2)  # 관절2 각도
    servo_joint1.ChangeDutyCycle(duty1)
    servo_joint2.ChangeDutyCycle(duty2)
    time.sleep(delay)
    servo_joint1.ChangeDutyCycle(0)
    servo_joint2.ChangeDutyCycle(0)

# 암 위로 올리기 시퀀스
def arm_up():
    # 관절1과 관절2를 순차적으로 위로 들어 올리는 동작
    # angle 리스트는 각도 증가 시퀀스를 정의하며,
    # 각 값은 servo_joint1, servo_joint2에 전달됨
    # 관절1: 아래 → 위 (70 → 150도)
    # 관절2: 관절1보다 약간 적게 움직임 (최대 145도까지)
    for angle in [70, 90, 110, 130, 150]:
        # 관절1에 angle 적용, 관절2는 마지막 단계에서 145로 제한
        move_servos(angle, angle if angle < 150 else 145)


# 암 내리기 시퀀스
def arm_down():
    # 관절1과 관절2를 순차적으로 아래로 내리는 동작
    # angle 리스트는 각도 감소 시퀀스를 정의함
    for angle in [140, 120, 100, 80, 60]:
        move_servos(angle, angle)  # 관절1, 2 동일 각도 적용

# 그립 동작
def grip_open():
    duty = angle_to_duty(20)
    servo_grip.ChangeDutyCycle(duty)
    time.sleep(0.2)
    servo_grip.ChangeDutyCycle(0)

def grip_close():
    duty = angle_to_duty(180)
    servo_grip.ChangeDutyCycle(duty)

# HTTP 요청 처리 클래스
class RequestHandler_httpd(BaseHTTPRequestHandler):
    def do_GET(self):
        message = b'Plz bee'
        self.send_response(200)
        self.send_header('Content-Type', 'text/plain')
        self.send_header('Content-Length', len(message))
        self.end_headers()
        self.wfile.write(message)

        # 요청 파싱
        request = self.requestline[5:-9]
        print("Request:", request)

        # 기본 PWM 시작
        servo_grip.start(0)
        servo_joint1.start(0)
        servo_joint2.start(0)

        # DC모터 제어 명령
        if request == '0':
            stop_dc_motors()
        elif request == '1':
            set_dc_motors([True, False, True, False, True, False, True, False])
        elif request == '2':
            set_dc_motors([False, True, False, True, False, True, False, True])
        elif request == '3':
            set_dc_motors([False, True, True, False, True, False, False, True])
        elif request == '4':
            set_dc_motors([True, False, False, True, False, True, False, True])

        # 암 제어
        elif request == '11':
            arm_up()
        elif request == '12':
            arm_down()

        # 그립 제어
        elif request == '21':
            grip_close()
        elif request == '22':
            grip_open()

# 서버 실행
server_address = ('192.168.82.193', 8080)
httpd = HTTPServer(server_address, RequestHandler_httpd)
print('Starting HTTP server...')
try:
    httpd.serve_forever()
except KeyboardInterrupt:
    print("\nShutting down server.")
    GPIO.cleanup()


| 구분               | 기능                                | 설명                   |
| ---------------- | --------------------------------- | -------------------- |
| **HTTP 서버**      | `1`, `2`, `11` 등 요청을 통해 동작을 제어 | 예)http://192.168.82.193:8080/1(문자열)                      |
| **DC모터**         | 8개 핀 사용                           | 특정 시퀀스에 따라 모터 제어     |
| **서보모터**         | 3개 (그립, 관절1, 관절2)                 | 각도 → 듀티비 변환으로 움직임 제어 |
| **암 동작**         | `11`, `12`                      | 여러 각도를 순차적으로 이동      |
| **그립 동작**        | `21`, `22`                      | 그립 오픈/클로즈            |
| **GPIO.cleanup** | 서버 종료 시 GPIO 초기화                  |                      |


| 리스트 인자 (`angle`)          | 사용 위치                          | 의미                  |
| ------------------------- | ------------------------------ | ------------------- |
| `[70, 90, 110, 130, 150]` | `arm_up()` → `move_servos()`   | 점차 위로 올리는 각도        |
| `[140, 120, 100, 80, 60]` | `arm_down()` → `move_servos()` | 점차 아래로 내리는 각도       |
| `angle1, angle2`          | `move_servos()` 내부             | 각각 관절1, 관절2에 적용될 각도 |


### 💻 리팩토링된 ROS2 ver(Python, manipulator_controller.py)

In [ ]:
import rclpy
from rclpy.node import Node
from std_msgs.msg import Int32
import RPi.GPIO as GPIO
import time

class ManipulatorController(Node):
    def __init__(self):
        super().__init__('manipulator_controller')

        # 토픽 구독: Int32 타입의 명령을 받음
        self.subscription = self.create_subscription(
            Int32,
            'arm_command',
            self.listener_callback,
            10
        )

        # GPIO 설정
        GPIO.setmode(GPIO.BCM)
        GPIO.setwarnings(False)

        # DC 모터 출력핀
        self.dc_pins = [0, 5, 6, 13, 19, 26, 16, 20]
        for pin in self.dc_pins:
            GPIO.setup(pin, GPIO.OUT)

        # 서보모터 출력핀
        self.servo1 = GPIO.PWM(10, 100)  # 그립
        self.servo2 = GPIO.PWM(9, 100)   # 관절1
        self.servo3 = GPIO.PWM(11, 100)  # 관절2

        for pin in [9, 10, 11]:
            GPIO.setup(pin, GPIO.OUT)

        self.servo1.start(0)
        self.servo2.start(0)
        self.servo3.start(0)

        self.get_logger().info('Manipulator controller node started')

    def listener_callback(self, msg):
        command = msg.data
        self.get_logger().info(f'Received command: {command}')

        if command == 0:
            self.stop_motors()
        elif command == 1:
            self.move_forward()
        elif command == 2:
            self.move_backward()
        elif command == 3:
            self.turn_left()
        elif command == 4:
            self.turn_right()
        elif command == 11:
            self.arm_up()
        elif command == 12:
            self.arm_down()
        elif command == 21:
            self.grip_on()
        elif command == 22:
            self.grip_off()

    def stop_motors(self):
        for pin in self.dc_pins:
            GPIO.output(pin, False)

    def move_forward(self):
        GPIO.output(0, True)
        GPIO.output(5, False)
        GPIO.output(6, True)
        GPIO.output(13, False)
        GPIO.output(19, True)
        GPIO.output(26, False)
        GPIO.output(16, True)
        GPIO.output(20, False)

    def move_backward(self):
        GPIO.output(0, False)
        GPIO.output(5, True)
        GPIO.output(6, False)
        GPIO.output(13, True)
        GPIO.output(19, False)
        GPIO.output(26, True)
        GPIO.output(16, False)
        GPIO.output(20, True)

    def turn_left(self):
        GPIO.output(0, False)
        GPIO.output(5, True)
        GPIO.output(6, True)
        GPIO.output(13, False)
        GPIO.output(19, True)
        GPIO.output(26, False)
        GPIO.output(16, False)
        GPIO.output(20, True)

    def turn_right(self):
        GPIO.output(0, True)
        GPIO.output(5, False)
        GPIO.output(6, False)
        GPIO.output(13, True)
        GPIO.output(19, False)
        GPIO.output(26, True)
        GPIO.output(16, False)
        GPIO.output(20, True)

    def angle_to_duty(self, angle):
        return ((angle * 0.01) + 0.5) * 10

    def move_servos(self, angle1, angle2, delay=0.2):
        duty1 = self.angle_to_duty(angle1)
        duty2 = self.angle_to_duty(angle2)
        self.servo2.ChangeDutyCycle(duty1)
        self.servo3.ChangeDutyCycle(duty2)
        time.sleep(delay)
        self.servo2.ChangeDutyCycle(0)
        self.servo3.ChangeDutyCycle(0)

    def arm_up(self):
        for angle in [70, 90, 110, 130, 150]:
            angle2 = angle if angle < 150 else 145
            self.move_servos(angle, angle2)

    def arm_down(self):
        for angle in [140, 120, 100, 80, 60]:
            self.move_servos(angle, angle)

    def grip_on(self):
        duty = self.angle_to_duty(180)
        self.servo1.ChangeDutyCycle(duty)
        time.sleep(0.2)
        self.servo1.ChangeDutyCycle(0)

    def grip_off(self):
        duty = self.angle_to_duty(20)
        self.servo1.ChangeDutyCycle(duty)
        time.sleep(0.2)
        self.servo1.ChangeDutyCycle(0)


def main(args=None):
    rclpy.init(args=args)
    node = ManipulatorController()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        print('Shutting down...')
    finally:
        GPIO.cleanup()
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    main()


### 💻 ROS 코드 테스트용 Topic 발행

In [ ]:
# 터미널1: ROS 2 노드 실행
ros2 run your_package_name manipulator_controller

# 터미널2: Int32 값으로 제어 명령 보내기
ros2 topic pub /arm_command std_msgs/msg/Int32 "data: 11"   # 암 올리기
ros2 topic pub /arm_command std_msgs/msg/Int32 "data: 22"   # 그립 OFF
ros2 topic pub /arm_command std_msgs/msg/Int32 "data: 0"    # 정지
